In [12]:
import numpy as np
import pandas as pd
import mario
from harpy import HarFileObj
import shutil
import scipy
import os
import re
import fnmatch
import importlib
import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, YEARS, RD_UNIT, DIMENSION, TECHS, OUTPUT_ROOT

In [13]:
AGG_FILES = {
    "EXIOBASE": "agg_base_to_rd/agg_EXIOBASE_3.8.22_to_rd.xlsx",
    "FIGARO": "agg_base_to_rd/agg_FIGARO_to_rd.xlsx",
    "OECD_ICIO": "agg_base_to_rd/agg_OECD_ICIO_to_rd.xlsx",
    "CFC": "agg_base_to_rd/agg_CFC_3.8.2_to_rd.xlsx",
    "GTAP": "agg_base_to_rd/agg_GTAPpost2007_to_rd.xlsx",
    "FDI": "agg_base_to_rd/agg_FDI_to_rd.xlsx",
    "Patents_Citation": "agg_base_to_rd/agg_Patents_Citation_to_rd.xlsx"
}

MAP_RD_FILES = {
    "EXIOBASE": "agg_rd_to_base/agg_rd_to_EXIOBASE.xlsx",
    "FIGARO": "agg_rd_to_base/agg_rd_to_FIGARO.xlsx",
    "OECD_ICIO": "agg_rd_to_base/agg_rd_to_OECD_ICIO.xlsx",
    "CFC": "agg_rd_to_base/agg_rd_to_CFC.xlsx",
    "GTAP": "agg_rd_to_base/agg_rd_to_GTAP.xlsx",
    "FDI": "agg_rd_to_base/agg_rd_to_FDI.xlsx",
    "Patents_Citation": "agg_rd_to_base/agg_rd_to_Patents_Citation.xlsx"
}


# ============================================================
# BASE -> available metrics mapping:
#   FDI -> "rd_dir_fdi"
#   Patents_Citation -> "rd_dir_cit"
#   EXIOBASE / FIGARO / OECD / GTAP -> "rd_embod_CI",   "rd_imp_CI",   "rd_dir_CI"
#   CFC -> "rd_embod_CFC", "rd_dir_CFC"
#   ALL -> computes every metric available for the chosen BASE (see ALL_METRICS in run())
#
# ============================================================

# What each metric means:
#   rd_dir_fdi(ps): direct R&D content transmitted through FDI ownership shares (R&D stock of the investing sector/region weighted by its
#                   share of foreign direct investment in the recipient).

#   rd_dir_cit: direct R&D content transmitted through patent citations (R&D stock of the cited sector/region weighted by the
#               share of citations it receives from the citing one).

#   rd_dir_CI: direct (first-tier) R&D content embodied in intermediate trade flows Z — R&D stock of the producing sector applied
#              directly to what it sells, with no supply-chain propagation.

#   rd_embod_CI: total embodied R&D content — rd_dir_CI propagated through the full production network (Leontief inverse), i.e. R&D
#                content coming from all upstream suppliers, domestic and foreign.

#   rd_imp_CI: same as rd_embod_CI but restricted to imported R&D content only (cross-border flows, domestic same-region flows zeroed out).`

#   rd_dir_CFC / rd_embod_CFC: same direct / embodied logic as rd_dir_CI / rd_embod_CI, but the R&D is transmitted through the
#                              capital stock (Kbar / consumption of fixed capital) instead of intermediate trade flows.

# ============================================================


METRICS_WANTED = "ALL"   # List of metrics to compute, or "ALL" for every metric available for BASE
PATH_WRK = "C:/Users/oscar/Desktop/ENTICE/rendu_fin_stage/"   # Root working directory: adjust depending on the machine / user


# ============================================================
# R&D DATA: choose between volume (constant prices) and value (current prices)
# ============================================================
#   "volume" -> RDQ files : rd_value_volume_M€  (deflated, constant prices, every country re-based on 2020)
#   "value"  -> RDV files : rd_value_M€          (current prices)
# RD_UNIT, DIMENSION, TECHS and OUTPUT_ROOT are defined in project_settings.py (imported in the first cell)

# ------------------------------------------------------------
# TECHNOLOGY DIMENSION
#   DIMENSION = "(c,s)"   -> R&D by country x sector only (no technology)
#   DIMENSION = "(c,s,t)" -> R&D by country x sector x technology: the stock and every metric are computed once per
#                            technology of TECHS (the MRIO base is parsed only once per year, the technologies share it)
# TECHS is either
#   - a list of technology codes / patterns, e.g. ["TJ11", "TJ07", "TJ01"]; "TG*" sums every code starting with TG;
#     the code of the file is the 4th block of the VAR column: RDQ_<country>_<sector>_<TECH>_<group>
#   - or a dict {name: [code or pattern, ...]} to group several codes in one technology, e.g. {"green": ["TG*"], "J": ["TJ*"]}
# The 3 codes below are only the 3 largest codes of the file (by R&D volume): replace them by your own list.
# The (c,s,t) files cover 2000-2021 and the manufacturing sectors (+ J62_J63): other years / sectors are not available.
# Each technology writes to its own folder: <OUTPUT_ROOT>/<technology>/<BASE>/output_loop_<BASE>_<year>/
# ------------------------------------------------------------

RD_FILES = {
    "volume_(c,s)": {"file": "RDQ (c,s).csv", "sep": ";", "col": "rd_value_volume_M€"},
    "volume_(c,s,t)": {"file": "RDQ (c,s,t).csv", "sep": ",", "col": None},
    "value_(c,s)": {"file": "RDV (c,s).csv", "sep": ",", "col": "rd_value_M€"},
    "value_(c,s,t)": {"file": "RDV (c,s,t).csv", "sep": ",", "col": None},
}

# sector code of the (c,s,t) files -> nace_code of the (c,s) files (and so of the agg_rd_to_* mappings)
TECH_SECTOR_MAP = {
    "S10": "C10_C12", "S13": "C13_C15", "S16": "C16_C18", "S19": "C19", "S20": "C20", "S21": "C21", "S22": "C22",
    "S23": "C23", "S24": "C24", "S25": "C25",
    "261": "C261", "262": "C262", "263": "C263", "264": "C264", "265": "C265", "266": "C266", "267": "C267", "268": "C268",
    "S27": "C27", "S28": "C28", "S29": "C29", "S30": "C30", "S31": "C31_C33",
    "S62": "J62_J63",
}
# OUTPUT_ROOT (one folder per unit x dimension, so results are never overwritten) comes from project_settings.py


# ============================================================
# PATHS: dictionary centralizing all input/output paths.
# Every "map_*" / "agg_*" key points to an Excel MAPPING file,
# i.e. a file that links the source base's raw nomenclature
# (regions/sectors) to the common working nomenclature
# (aggregated regions/sectors used across all bases).
# THESE are the files to adapt/recreate if the aggregation
# nomenclature changes or if a new database is added.
#
# Which bases are auto-downloaded vs. must be downloaded manually:
#   - EXIOBASE : downloaded automatically by mario (mario.download_exiobase3)
#                into PATHS["delete"], for each requested year. Nothing to
#                download by hand.
#   - CFC      : reuses EXIOBASE under the hood (same auto-download as above)
#                for the IOT part, but the Kbar capital matrices (PATHS["kbar"])
#                must be downloaded/placed manually, one .mat file per year.
#   - FIGARO   : downloaded automatically by mario (mario.parse_figaro
#                downloads + parses in one call) into PATHS["delete"]. Nothing
#                to download by hand.
#   - OECD_ICIO: NOT auto-downloaded. The OECD ICIO CSV files must be
#                downloaded manually and placed in PATHS["delete"]/2016-2022_EXT/
#                (one CSV per year); mario only parses them locally.
#   - GTAP     : NOT auto-downloaded. The GTAP MRIO files (per period) must be
#                downloaded manually and placed under PATHS["gtap_folder"]
#                (see parse_gtap_for_year for the exact per-year subfolder);
#                mario only parses them locally (mario.parse_gtap).
#   - FDI / Patents_Citation : not MRIO databases, just static local files
#                (PATHS["har"] / PATHS["citation"]) that must already be on disk.
# ============================================================
PATHS = {
    # --- R&D data (country/sector/year panel): used for all bases ---
    "rd": PATH_WRK + RD_FILES[f"{RD_UNIT}_{DIMENSION}"]["file"],
    # R&D mapping: Excel file with 2 sheets
    #   - "Sector": columns 'nace_code' (source sector code) -> 'aggregation' (target aggregated sector)
    #   - "Region": columns 'country_code' (source country code) -> 'aggregation' (target aggregated region)
    "agg_rd": PATH_WRK + f"mapping/{MAP_RD_FILES[BASE]}",
    #f"agg_rd_{BASE}": PATH_WRK + f"mapping/agg_rd_to_base/agg_rd_to_{BASE}.xlsx",

    # --- Specific to FDI ---
    # Raw .har file containing the FDIP matrix (source x host x sector)
    "har": PATH_WRK + "data_raw/data_FDI/fdi_158_65.har",

    # --- Specific to Citation (patents) ---
    # Raw CSV of patent citations (citing/cited country and sector)
    "citation": PATH_WRK + "data_raw/data_Patents_Citation/Fractional_Citations_6_yearlyCohorts_Of_CitingFam_backward_2012-2023.prefiltered.csv",

    # --- Specific to MRIO (EXIOBASE / FIGARO / OECD) ---
    "delete": PATH_WRK + f"data_raw/data_{BASE}",     # temporary download/parsing folder (deleted after each year)
    # MRIO -> working nomenclature aggregation mapping: Excel file with 'Region' and 'Sector' sheets
    # (source columns / 'aggregation' column, in the format expected by mario's db.aggregate())
    # "agg": PATH_WRK + "mapping/agg_GTAP1.xlsx",
    # Test
    "agg": PATH_WRK + f"mapping/{AGG_FILES[BASE]}",


    # --- Specific to GTAP (static base, one parquet file per period, no yearly download) ---
    "gtap_folder": PATH_WRK + "GTAP12.Power.MRIO_mario_parquet",
    "agg_gtap": PATH_WRK + "mapping/agg_base_to_rd/agg_GTAPpost2007_to_rd.xlsx",       # GTAP aggregation mapping used for year > 2007
    "agg_gtap2": PATH_WRK + "mapping/agg_base_to_rd/agg_GTAPpre2007_to_rd.xlsx",  # GTAP aggregation mapping used for year <= 2007

    # --- Specific to CFC ---
    # .mat file of the capital matrix Kbar; {year} is filled in automatically by load_kbar_for_year()
    "kbar": PATH_WRK + "data_raw/all_kbar/Kbar_exio_v3_8_2_{year}_cfc_pxi.mat",
    # Kbar -> working nomenclature aggregation mapping: 'Region' & 'Sector' sheets
    "agg_kb": PATH_WRK + "mapping/agg_base_to_rd/agg_Kbar_to_rd.xlsx",
    # Product -> industry mapping (columns: product code, industry code), needed to convert Kbar from prod x ind to ind x ind
    "prod_to_ind": PATH_WRK + "mapping/Mapping_Ind_prod.xlsx",
    # ISO3 -> ISO2 mapping for Kbar countries: sheet 'ISO', columns 'iso3' -> 'iso2'
    "map_cfc_iso": PATH_WRK + "mapping/map_cfc_iso.xlsx",

    # --- GTAP region/sector -> R&D nomenclature mappings, used for FDI and Citation ---
    # "map_fdi": sheet 'Region' (columns 'gtap_region' -> 'rd_region') and sheet 'Sector' (columns 'gtap_sector' -> 'nace_sector')
    "map_fdi": PATH_WRK + f"mapping/{AGG_FILES[BASE]}",
    # "map_citation": sheet 'Sector' (columns 'citation_code' -> 'nace_code') and sheet 'Region' (columns 'citation_region' -> 'rd_region')
    "map_citation": PATH_WRK + f"mapping/{AGG_FILES[BASE]}",

    # --- Output file/folder ---
    # Root folder where results are written (one subfolder per year + the global exports/imports/production files)
    "output": PATH_WRK + f"{OUTPUT_ROOT}/{BASE}/output_loop_{BASE}",
}



VERSION_EXIO = "3.8.2"   # EXIOBASE version to download, used only if BASE == 'EXIOBASE' or 'CFC'

Y0, YN, DEP = 1995, 2022, 0.15   # Time bounds and depreciation rate used to compute the R&D stock (compute_rd_stock)


In [14]:
def load_mapping(path, sheet, key_col, val_col):
    """Loads a key->value mapping from an Excel worksheet.

    Parameters

    path : path to the Excel file
    sheet : name of the worksheet
    key_col : name of the source column (dictionary keys)
    val_col : name of the target column (dictionary values)

    Returns

    dict {key_col -> val_col}
    """

    df = pd.read_excel(path, sheet_name=sheet, dtype=str)
    return dict(zip(df[key_col], df[val_col]))

In [15]:
## R&D stock
# df_rd, stock = compute_rd_stock(paths["rd"], paths["agg_rd"])                    # without technology
# df_rd, stock = compute_rd_stock(paths["rd"], paths["agg_rd"], tech=["TJ11"])     # one technology of the (c,s,t) file


def tech_specs(techs):
    """TECHS -> {label: [patterns]} (label = name of the output sub-folder)."""
    if isinstance(techs, dict):
        specs = {k: ([v] if isinstance(v, str) else list(v)) for k, v in techs.items()}
    else:
        specs = {t: [t] for t in techs}
    return {re.sub(r"[^A-Za-z0-9_+-]+", "x", str(k)): v for k, v in specs.items()}


def output_path(paths, label=None):
    """Output folder (without the _<year> suffix): paths["output"] without technology, one sub-folder per technology otherwise."""
    if label is None:
        return paths["output"]
    return PATH_WRK + f"{OUTPUT_ROOT}/{label}/{BASE}/output_loop_{BASE}"


def _tag(label):
    return "" if label is None else f" [{label}]"


def _store(output, year, label, metrics):
    if label is None:
        output[year] = metrics
    else:
        output.setdefault(year, {})[label] = metrics


def load_rd_flows(path_rd, spec, tech=None):
    """Reads the R&D flow file and returns a long DataFrame [year, country_code, nace_code, rd_value].
    tech=None : (c,s) file (long format).
    tech=list of patterns : (c,s,t) file (wide format, one row per VAR = RDQ_<country>_<sector>_<tech>_<group>, one column per
    year); keeps the technologies matching a pattern and sums them (and their groups) by country x sector x year."""
    raw = pd.read_csv(path_rd, sep=spec["sep"])
    if tech is None:
        rd = raw[["year", "country_code", "nace_code", spec["col"]]].rename(columns={spec["col"]: "rd_value"})
        return rd

    parts = raw["VAR"].str.split("_", expand=True)
    tech_code = parts[3]
    keep = tech_code.map(lambda code: any(fnmatch.fnmatchcase(code, p) for p in tech))
    if not keep.any():
        raise ValueError(f"No technology of the file matches {tech}. Available codes: {sorted(tech_code.unique())}")
    raw = raw[keep.to_numpy()].copy()
    raw["country_code"] = parts.loc[keep, 1].to_numpy()
    sector_code = parts.loc[keep, 2]
    raw["nace_code"] = sector_code.map(TECH_SECTOR_MAP).to_numpy()
    unknown = sorted(sector_code[raw["nace_code"].isna().to_numpy()].unique())
    if unknown:
        print(f"  [warning] sector codes without nace_code in TECH_SECTOR_MAP (dropped): {unknown}")
    raw = raw.dropna(subset=["nace_code"])

    year_cols = [c for c in raw.columns if re.fullmatch(r"\d{4}Y1", str(c))]
    long = raw.melt(id_vars=["country_code", "nace_code"], value_vars=year_cols, var_name="ycol", value_name="rd_value")
    long["year"] = long["ycol"].str[:4].astype(int)
    return long.groupby(["year", "country_code", "nace_code"], as_index=False)["rd_value"].sum(min_count=1)


def compute_rd_stock(path_rd, path_agg_rd, rd_unit=None, y0=Y0, yn=YN, dep=DEP, tech=None):
    """ Reads the CSV file containing the R&D flow data, performs the mapping
    and calculates the R&D stock for all years with the permanent inventory method.
    With tech (list of patterns) the flows of the (c,s,t) file are used: the stock is then computed over the years of that
    file (first / last year instead of y0 / yn) and a missing flow counts as 0."""

    spec = RD_FILES[rd_unit or f"{RD_UNIT}_{DIMENSION}"]   # file, separator and value column depend on the chosen unit / dimension
    rd_vector = load_rd_flows(path_rd, spec, tech)

    map_nace = pd.read_excel(path_agg_rd, sheet_name="Sector", dtype=str)
    map_country = pd.read_excel(path_agg_rd, sheet_name="Region", dtype=str)

    dict_nace = dict(zip(map_nace["nace_code"], map_nace["aggregation"]))
    dict_country = dict(zip(map_country["country_code"], map_country["aggregation"]))

    rd_vector["nace_code"] = rd_vector["nace_code"].map(dict_nace)
    rd_vector["country_code"] = rd_vector["country_code"].map(dict_country)

    data_rd_agg = (
        rd_vector.groupby(["year", "country_code", "nace_code"], as_index=False)["rd_value"].sum()
    )
    df_rd = data_rd_agg.pivot_table(index="year", columns=["country_code", "nace_code"], values="rd_value")
    df_rd.columns.names = ["Region", "Sector"]

    if tech is not None:
        df_rd = df_rd.fillna(0.0)
        y0, yn = int(df_rd.index.min()), int(df_rd.index.max())

    time = df_rd.index
    stock = pd.DataFrame(0.0, index=df_rd.index, columns=df_rd.columns)

    for col in df_rd.columns:
        flux = df_rd[col]
        f0, fn = flux.loc[y0], flux.loc[yn]

        if pd.isna(f0) or pd.isna(fn) or f0 == 0 or fn == 0:
            tx = 0
        else:
            tx = (fn / f0) ** (1 / (yn - y0)) - 1

        tx = max(tx, 0)
        tx = min(tx, 0.1)

        stock.loc[y0, col] = f0 / (dep + tx)

        for t in time:
            if t <= y0:
                stock.loc[t, col] = stock.loc[y0, col]
            else:
                stock.loc[t, col] = (1 - dep) * stock.loc[t - 1, col] + flux.loc[t]

    return df_rd, stock


def e_rd_for_year(stock, year, target_index):
    """Retrieves the desired year from the `stock` array and aligns it with
    the target index. Returns the R&D vector (Series) — use
    `M.multiply(e_rd, axis=0)` to reproduce what `e_hat @ M` used to do."""

    e_rd = stock.loc[[year]].copy()
    e_rd.columns = pd.MultiIndex.from_arrays(
        [e_rd.columns.get_level_values("Region"), e_rd.columns.get_level_values("Sector")],
        names=["Region", "Sector"],
    )
    e_rd_aligned = e_rd.reindex(columns=target_index, fill_value=0)
    return pd.Series(e_rd_aligned.values.flatten(), index=target_index)


In [16]:
## FDI

def load_fdi_static(path_har, dict_reg_gtap_to_rd, gtap_to_nace_rd_data):
    """ Reads ans maps FDI databases."""

    har = HarFileObj.loadFromDisk(path_har)
    headers = har.getHeaderArrayObjs(har.getHeaderArrayNames())
    fdip_obj = [o for o in headers if o["name"] == "FDIP"][0]
    reg = fdip_obj["sets"][0]["dim_desc"]
    comm = fdip_obj["sets"][2]["dim_desc"]
    arr = fdip_obj["array"]

    src_idx, hst_idx, sec_idx = np.nonzero(arr > 0)
    df_fdip = pd.DataFrame({
        "source": [reg[i] for i in src_idx],
        "host": [reg[j] for j in hst_idx],
        "sector": [comm[k] for k in sec_idx],
        "FDIP": arr[src_idx, hst_idx, sec_idx],
    })

    df_fdip_pivot = (
        df_fdip.pivot(index=["host", "sector"], columns="source", values="FDIP")
        .reset_index()
    )
    df_fdip_pivot.columns.name = None
    df_fdip_pivot = df_fdip_pivot.fillna(0).set_index(["host", "sector"]).rename_axis(
        index=["Region", "Sector"]
    )

    df_tmp = df_fdip_pivot.copy()
    df_tmp.columns = df_tmp.columns.map(dict_reg_gtap_to_rd)
    df_tmp = df_tmp.T.groupby(level=0).sum().T

    df_tmp = (
        df_tmp.reset_index()
        .assign(Region=lambda x: x["Region"].map(dict_reg_gtap_to_rd))
        .groupby(["Region", "Sector"]).sum()
    )

    df_tmp = (
        df_tmp.reset_index()
        .assign(Sector=lambda x: x["Sector"].map(gtap_to_nace_rd_data))
        .groupby(["Region", "Sector"]).sum()
    )

    fdip_reg = df_tmp.sum()
    fdip_reg_safe = np.where(fdip_reg == 0, np.nan, fdip_reg)
    fdips_reg = df_tmp.div(fdip_reg_safe, axis=1)

    return fdips_reg


def metric_fdi_for_year(fdips_reg, stock, year):
    e_rd = e_rd_for_year(stock, year, fdips_reg.index)
    return {"rd_dir_fdips": fdips_reg.multiply(e_rd, axis=0)}


In [17]:
## Patent citation


def load_citation_static(path_csv, dict_sect, dict_reg, year):
    """ Reads the raw patent citation CSV, maps countries/sectors to the
    working nomenclature, filters out 'rest of world' (RW) and unmapped
    sectors, then builds the square matrix of citation shares (normalized
    per column) for the given year. """

    df = pd.read_csv(path_csv)

    # Strict conversion of types to strings
    df["Cited_nace2_code"] = df["Cited_nace2_code"].astype(str).str.strip()
    df["Citing_nace2_code"] = df["Citing_nace2_code"].astype(str).str.strip()
    df["Cited_Inv_Country"] = df["Cited_Inv_Country"].astype(str).str.strip()
    df["Citing_Inv_Country"] = df["Citing_Inv_Country"].astype(str).str.strip()

    # Cleaning of the mapping dictionaries
    cleaned_dict_sect = {
        str(k).strip(): str(v).strip()
        for k, v in dict_sect.items()
        if pd.notna(k) and pd.notna(v)
    }
    cleaned_dict_reg = {
        str(k).strip(): str(v).strip()
        for k, v in dict_reg.items()
        if pd.notna(k) and pd.notna(v)
    }

    #  Direct mapping on the original DataFrame's columns
    df["Cited_Inv_Country"] = (
        df["Cited_Inv_Country"].map(cleaned_dict_reg).fillna("RW")
    )
    df["Citing_Inv_Country"] = (
        df["Citing_Inv_Country"].map(cleaned_dict_reg).fillna("RW")
    )

    df["Cited_nace2_code"] = df["Cited_nace2_code"].map(cleaned_dict_sect)
    df["Citing_nace2_code"] = df["Citing_nace2_code"].map(cleaned_dict_sect)

    #  Strict filtering: removal of unmapped sectors (NaN) and RW countries
    df_filtered = df.dropna(subset=["Cited_nace2_code", "Citing_nace2_code"])
    df_filtered = df_filtered[
        (df_filtered["Cited_Inv_Country"] != "RW")
        & (df_filtered["Citing_Inv_Country"] != "RW")
    ]

    #  Final pivot
    df_cit = pd.pivot_table(
        df_filtered,
        index=["Cited_Inv_Country", "Cited_nace2_code"],
        columns=["Citing_Inv_Country", "Citing_nace2_code"],
        values="fr_nr_citations",
        aggfunc="sum",
        fill_value=0,
    )

    #  Forcing type homogeneity of the index and columns
    index_pairs = [(str(c[0]), str(c[1])) for c in df_cit.index]
    column_pairs = [(str(c[0]), str(c[1])) for c in df_cit.columns]

    df_cit.index = pd.MultiIndex.from_tuples(index_pairs, names=["Region", "Sector"])
    df_cit.columns = pd.MultiIndex.from_tuples(
        column_pairs, names=["Region", "Sector"]
    )

    #  Balancing the perfectly square matrix
    all_pairs = sorted(set(index_pairs) | set(column_pairs))
    df_cit = df_cit.reindex(index=all_pairs, columns=all_pairs, fill_value=0)

    #  Normalization (computing citation shares per column)
    df_cited = df_cit.sum().copy()
    df_cited_safe = np.where(df_cited == 0, np.nan, df_cited)
    df_shares = df_cit.div(df_cited_safe, axis=1).fillna(0)

    # === ADDING THE YEAR DIMENSION TO THE ROW INDEX ===
    # Insert the current year as the top level of the row index
    df_shares.index = pd.MultiIndex.from_tuples(
        [(int(year), idx[0], idx[1]) for idx in df_shares.index],
        names=["Year", "Region", "Sector"]
    )

    return df_shares

def metric_citation_for_year(df_shares, stock, year):
    """ Combines the citation shares (df_shares) with the R&D stock for the
    given year to produce the rd_dir_cit metric (direct R&D content
    transmitted via patent citations). """

    target_idx_2d = df_shares.index.droplevel("Year")
    e_rd = e_rd_for_year(stock, year, target_idx_2d)
    e_rd.index = df_shares.index

    return {"rd_dir_cit": df_shares.multiply(e_rd, axis=0)}

In [18]:
def parse_mrio_for_year(base, year, paths, version=None):
    """ Downloads (if needed) and parses the requested MRIO base (FIGARO,
    EXIOBASE or OECD_ICIO) for the given year via mario, then aggregates it
    using the paths["agg"] mapping. Returns the aggregated base object. """
    if base == "FIGARO":
        db = mario.parse_figaro(table="IOT", year=year, iot_mode="industry", path=PATHS["delete"])
    elif base == "EXIOBASE":
        mario.download_exiobase3(path=PATHS["delete"], years=[year], table="IOT", system="ixi", version=version, overwrite=True)
        db = mario.parse_exiobase(path=f"{PATHS['delete']}/IOT_{year}_ixi", table="IOT", unit="Monetary")
    elif base == "OECD_ICIO":
        db = mario.parse_oecd(path=f"{PATHS['delete']}/2016-2022_EXT/{year}.csv", year=year)

    else:
        raise ValueError(f"MRIO database unmanaged : {base}")

    db.aggregate(paths["agg"], ignore_nan=True)
    return db



def build_rd_extensions_for_year(stock, df_rd, year, E_columns, units_template):
    """ Builds the new Satellite accounts (stock + flow) to add to the MRIO
    base's E matrix for the given year, aligned on E_columns with the
    corresponding units (M€). """
    stock_rd_i = stock.loc[[year]].rename(index={year: "R&D_stock"})
    flux_rd_i = df_rd.loc[[year]].rename(index={year: "R&D_flux"})
    rd_extensions = pd.concat([stock_rd_i, flux_rd_i], axis=0)

    rd_extensions.columns = pd.MultiIndex.from_arrays(
        [
            rd_extensions.columns.get_level_values("Region"),
            ["Sector"] * len(rd_extensions.columns),
            rd_extensions.columns.get_level_values("Sector"),
        ],
        names=["Region", "Level", "Item"],
    )
    rd_extensions.columns.names = E_columns.names
    rd_extensions = rd_extensions.reindex(columns=E_columns, fill_value=0)

    new_units = units_template.iloc[:1].copy()
    new_units = pd.concat([new_units, new_units])
    new_units.index = ["R&D_stock", "R&D_flux"]
    new_units.loc["R&D_stock", "unit"] = "M€"
    new_units.loc["R&D_flux", "unit"] = "M€"

    return rd_extensions, new_units


def compute_mrio_metrics(extended_db, metrics_wanted, w=None):
    """Calculates only the requested metrics (avoids unnecessary calculations,
    e.g. e_hat @ w if only RD_CI_dir is required). w: Leontief inverse already computed (it does not depend on the R&D
    stock, so it is shared between technologies)."""

    results = {}
    e_rd = extended_db.e.loc["R&D_stock"].copy()

    if "rd_embod_CI" in metrics_wanted or "rd_imp_CI" in metrics_wanted:
        f = (extended_db.w if w is None else w).multiply(e_rd, axis=0)
        if "rd_embod_CI" in metrics_wanted:
            results["rd_embod_CI"] = f @ extended_db.Z
        if "rd_imp_CI" in metrics_wanted:
            Z_imp = extended_db.Z.copy()
            reg_prod = np.array(Z_imp.index.get_level_values("Region"))
            reg_cons = np.array(Z_imp.columns.get_level_values("Region"))
            mask = reg_prod[:, None] == reg_cons[None, :]
            Z_imp = pd.DataFrame(np.where(mask, 0, Z_imp.values), index=Z_imp.index, columns=Z_imp.columns)
            results["rd_imp_CI"] = f @ Z_imp

    if "rd_dir_CI" in metrics_wanted:
        results["rd_dir_CI"] = extended_db.Z.multiply(e_rd, axis=0)
    if "intensity_vector" in metrics_wanted:
         results["intensity_vector"] = e_rd

    return results


In [19]:
def compute_exports_imports_by_sector(db):
    """ From an MRIO base (db), computes for each (Region, Sector): exports,
    intermediate imports, final-demand imports (per region), and
    production. Returns 4 DataFrames. """
    Z = db.query(matrices="Z", scenarios="baseline")
    Y = db.query(matrices="Y", scenarios="baseline")

    row_regions = Z.index.get_level_values("Region").to_numpy()      # Z and Y share the same row index
    col_regions_Z = Z.columns.get_level_values("Region").to_numpy()
    col_regions_Y = Y.columns.get_level_values("Region").to_numpy()

    Z_vals, Y_vals = Z.to_numpy(), Y.to_numpy()

    # --- Exports by (Region, Sector) ---
    mask_Z = row_regions[:, None] != col_regions_Z[None, :]   # True where seller's region != row's region
    mask_Y = row_regions[:, None] != col_regions_Y[None, :]

    exp_Z = np.where(mask_Z, Z_vals, 0).sum(axis=1)
    exp_Y = np.where(mask_Y, Y_vals, 0).sum(axis=1)

    exports = pd.Series(exp_Z + exp_Y, index=Z.index, name="exports")
    exports = exports.reset_index()
    exports = exports.drop(columns="Level").rename(columns={"Item": "Sector"})

    # --- Imports (intermediate consumption) by (Region, Sector) ---
    # même mask_Z que pour exports, mais somme sur l'autre axe (colonnes -> import) au lieu des lignes (-> export)
    imports_intermediate = pd.Series(
        np.where(mask_Z, Z_vals, 0).sum(axis=0), index=Z.columns, name="imports_intermediate"
    )
    imports_intermediate = imports_intermediate.reset_index()
    imports_intermediate = imports_intermediate.drop(columns="Level").rename(columns={"Item": "Sector"})

    # --- Imports (final demand) by Region only ---
    # block[a, b] = somme de Y pour les lignes de la region a et les colonnes de la region b
    block = (
        pd.DataFrame(Y_vals, index=row_regions, columns=col_regions_Y)
        .groupby(level=0).sum()
        .T.groupby(level=0).sum().T
    )
    col_totals = block.sum(axis=0)                                          # total colonne (toutes regions confondues)
    domestic = pd.Series(np.diag(block.reindex(block.columns)), index=block.columns)  # part "domestique" (a == b)
    imports_final_demand = (col_totals - domestic).rename("imports_final_demand")
    imports_final_demand = imports_final_demand.reset_index().rename(columns={"index": "Region"})

    # --- Production (X) by (Region, Sector) ---
    production = db.X.copy()
    production.name = "production"
    production = production.reset_index()
    production = production.drop(columns="Level").rename(columns={"Item": "Sector"})

    return exports, imports_intermediate, imports_final_demand, production
# Global lists accumulating the results (exports/imports/production) for every year
# processed in the run() loop, so they can be saved into a single pivoted file at the end.
all_exports = []
all_imports_inter = []
all_imports_final = []
all_productions = []

def _save_trade(exports, imports_intermediate, imports_final_demand, production, base, year):
    """Adds the year to each DataFrame and appends it to the global lists."""
    # Add the year column to each DataFrame

    exports = exports.copy()
    exports["year"] = year
    all_exports.append(exports)

    imports_intermediate = imports_intermediate.copy()
    imports_intermediate["year"] = year
    all_imports_inter.append(imports_intermediate)

    imports_final_demand = imports_final_demand.copy()
    imports_final_demand["year"] = year
    all_imports_final.append(imports_final_demand)

    production = production.copy()
    production["year"] = year
    all_productions.append(production)

In [20]:
def load_kbar_for_year(year, paths, mapping_iso3_to_iso2):
    """Loads and formats the capital matrix Kbar from the .mat file,
    converts it to ISO2 format, converts prod x ind to ind x ind,
    then aggregates it into the working bill of materials."""

    data = scipy.io.loadmat(paths["kbar"].format(year=year))

    reg_kbar        = [str(c[0]).strip() for c in data["countries"].flatten()]
    ind_labels_kbar = [str(i[0]).strip() for i in data["indLabels"].flatten()]
    prod_labels_kbar= [str(p[0]).strip() for p in data["prodLabels"].flatten()]
    Kbar            = data["Kbar"]

    col_index = pd.MultiIndex.from_product(
        [reg_kbar, ["Sector"], ind_labels_kbar],
        names=["Region", "Level", "Item"],
    )
    row_index = pd.MultiIndex.from_product(
        [reg_kbar, ["Sector"], prod_labels_kbar],
        names=["Region", "Level", "prod"],
    )
    df_cap = pd.DataFrame(Kbar, index=row_index, columns=col_index)

    # ISO3 to ISO2
    df_cap.index   = df_cap.index.set_levels(
        df_cap.index.levels[0].map(mapping_iso3_to_iso2), level=0
    )
    df_cap.columns = df_cap.columns.set_levels(
        df_cap.columns.levels[0].map(mapping_iso3_to_iso2), level=0
    )

    # Conversion prod x ind to ind x ind
    map_prod_to_ind = pd.read_excel(paths["prod_to_ind"], sheet_name="Mapping")
    prod_col        = map_prod_to_ind.columns[0]
    ind_col         = map_prod_to_ind.columns[1]
    mapping_p2i     = dict(zip(map_prod_to_ind[prod_col], map_prod_to_ind[ind_col]))

    df_reset = df_cap.reset_index()
    df_reset["ind_mapped"] = df_reset["prod"].map(mapping_p2i)
    df_reset = df_reset.drop(columns="prod")
    df_cap_mapped = df_reset.groupby(["Region", "Level", "ind_mapped"], sort=False).sum()
    df_cap_mapped.index.names = ["Region", "Level", "Item"]

    # Aggregation into the working bill of materials
    map_sec = pd.read_excel(paths["agg_kb"], sheet_name="Sector", dtype=str)
    map_reg = pd.read_excel(paths["agg_kb"], sheet_name="Region", dtype=str)

    dict_sec = dict(zip(map_sec["Item"],   map_sec["Aggregation"]))
    dict_reg = dict(zip(map_reg["Region"], map_reg["Aggregation"]))

    # Mapping rows
    idx = df_cap_mapped.index.to_frame(index=False)
    idx["Item"]   = idx["Item"].map(dict_sec).fillna(idx["Item"])
    idx["Region"] = idx["Region"].map(dict_reg).fillna(idx["Region"])
    df_cap_mapped.index = pd.MultiIndex.from_frame(idx)

    # Mapping columns
    col = df_cap_mapped.columns.to_frame(index=False)
    col["Item"]   = col["Item"].map(dict_sec).fillna(col["Item"])
    col["Region"] = col["Region"].map(dict_reg).fillna(col["Region"])
    df_cap_mapped.columns = pd.MultiIndex.from_frame(col)

    # Final aggregation
    df_cap_mapped = df_cap_mapped.groupby(level=["Region", "Level", "Item"]).sum()
    df_cap_mapped = df_cap_mapped.T.groupby(level=["Region", "Level", "Item"]).sum().T

    return df_cap_mapped


def compute_cfc_metrics(db, extended_db, df_cap_mapped, stock, year, metrics_wanted, cache=None):

    """Metrics specific to the CFC database.

    Calculated matrices:
      - K_aligned : capital intensity (Kbar normalised by output X)
      - L_Ak : Leontief inverse extended to capital  (I - (z + K))^-1
      - f_Ak : e_hat @ L_Ak
      - F_CFC_rd_Ak : f_Ak @ Kbar   (indirect R&D content via capital + CI)
      - CFC_direct : e_hat @ K_aligned   (direct R&D content via capital)
    """

    results = {}

    cache = {} if cache is None else cache
    if "K_aligned" not in cache:   # independent of the R&D stock: computed once per year, shared between technologies
        X = db.X.reindex(df_cap_mapped.columns)
        X_values = X.values.flatten()
        X_safe = np.where(X_values == 0, np.nan, X_values)
        K = df_cap_mapped.div(X_safe, axis=1)

        K_aligned = K.reindex(index=db.z.index, columns=db.z.columns).fillna(0)

        I    = pd.DataFrame(np.eye(len(db.z)), index=db.z.index, columns=db.z.columns)
        L_Ak = pd.DataFrame(
            np.linalg.inv((I - (db.z + K_aligned)).values),
            index=db.z.index, columns=db.z.columns,
        )
        cache["K_aligned"], cache["L_Ak"] = K_aligned, L_Ak
    K_aligned, L_Ak = cache["K_aligned"], cache["L_Ak"]

    e_rd = extended_db.e.loc["R&D_stock"].copy()

    if "rd_embod_CFC" in metrics_wanted:
        Kbar_aligned = df_cap_mapped.reindex(index=K_aligned.columns, columns=K_aligned.columns, fill_value=0)
        f_Ak = L_Ak.multiply(e_rd, axis=0)                 # remplace e_hat @ L_Ak
        results["rd_embod_CFC"] = f_Ak @ Kbar_aligned

    if "rd_dir_CFC" in metrics_wanted:
        results["rd_dir_CFC"] = K_aligned.multiply(e_rd, axis=0)   # remplace e_hat @ K_aligned

    return results

In [21]:
# %%
def parse_gtap_for_year(year, paths):
    """
    Parses and aggregates the GTAP dataset specifically for the projected year 'year'.
    Applies the right aggregation file depending on whether year > 2007 or not.
    """

    db_folder = PATH_WRK + f"data_raw/V12APOW/V12APOW_MRIO_{year}/"

    db = mario.parse_gtap(
        path=db_folder,
        table="IOT",
        input_format="csv",
        calc_all=False,
    )

    if year > 2007:
        agg_file = paths["agg_gtap"]
    else:
        agg_file = paths["agg_gtap2"]

    db.aggregate(agg_file, ignore_nan=True)
    return db


def _write_global_files(final_folder, base):
    """Pivots the exports / imports / production accumulated over the years (all_* lists) and writes them (one file per
    variable, years as columns) in final_folder; years already in an existing file are replaced, the others kept."""
    os.makedirs(final_folder, exist_ok=True)

    df_exports_raw = pd.concat(all_exports, ignore_index=True)
    df_imports_inter_raw = pd.concat(all_imports_inter, ignore_index=True)
    df_imports_final_raw = pd.concat(all_imports_final, ignore_index=True)
    df_production_raw = pd.concat(all_productions, ignore_index=True)

    df_exports_pivoted = df_exports_raw.pivot(index=["Region", "Sector"], columns="year", values="exports").reset_index()
    df_exports_pivoted.columns.name = None

    df_imports_inter_pivoted = df_imports_inter_raw.pivot(index=["Region", "Sector"], columns="year", values="imports_intermediate").reset_index()
    df_imports_inter_pivoted.columns.name = None

    df_imports_final_pivoted = df_imports_final_raw.pivot(index="Region", columns="year", values="imports_final_demand").reset_index()
    df_imports_final_pivoted.columns.name = None

    df_production_pivoted = df_production_raw.pivot(index=["Region", "Sector"], columns="year", values="production").reset_index()
    df_production_pivoted.columns.name = None

    if not os.path.exists(f"{final_folder}exports_{base}.csv"):
        df_exports_pivoted.to_csv(f"{final_folder}exports_{base}.csv", sep=";", index=False)
        df_imports_inter_pivoted.to_csv(f"{final_folder}imports_intermediate_{base}.csv", sep=";", index=False)
        df_imports_final_pivoted.to_csv(f"{final_folder}imports_final_demand_{base}.csv", sep=";", index=False)
        df_production_pivoted.to_csv(f"{final_folder}production_{base}.csv", sep=";", index=False)
    else:
        df_exports_incomplete = pd.read_csv(f"{final_folder}exports_{base}.csv", sep=";")
        df_imports_inter_incomplete = pd.read_csv(f"{final_folder}imports_intermediate_{base}.csv", sep=";")
        df_imports_final_incomplete = pd.read_csv(f"{final_folder}imports_final_demand_{base}.csv", sep=";")
        df_production_incomplete = pd.read_csv(f"{final_folder}production_{base}.csv", sep=";")
        df_exports_pivoted.columns = df_exports_pivoted.columns.astype(str)
        df_imports_inter_pivoted.columns = df_imports_inter_pivoted.columns.astype(str)
        df_imports_final_pivoted.columns = df_imports_final_pivoted.columns.astype(str)
        df_production_pivoted.columns = df_production_pivoted.columns.astype(str)
        df_exports_incomplete = df_exports_incomplete.merge(
        df_exports_pivoted, on=["Region", "Sector"], how="outer", suffixes=("_old", "")
        )
        df_imports_inter_incomplete = df_imports_inter_incomplete.merge(
        df_imports_inter_pivoted, on=["Region", "Sector"], how="outer", suffixes=("_old", "")
        )
        df_imports_final_incomplete = df_imports_final_incomplete.merge(
        df_imports_final_pivoted, on=["Region"], how="outer", suffixes=("_old", "")
        )
        df_production_incomplete = df_production_incomplete.merge(
        df_production_pivoted, on=["Region", "Sector"], how="outer", suffixes=("_old", "")
        )
        cols_to_drop = [c for c in df_exports_incomplete.columns if c.endswith("_old")]
        df_exports_incomplete = df_exports_incomplete.drop(columns=cols_to_drop)
        df_imports_inter_incomplete = df_imports_inter_incomplete.drop(columns=cols_to_drop)
        df_imports_final_incomplete = df_imports_final_incomplete.drop(columns=cols_to_drop)
        df_production_incomplete = df_production_incomplete.drop(columns=cols_to_drop)
        df_exports_incomplete.to_csv(f"{final_folder}exports_{base}.csv", sep=";", index=False)
        df_imports_inter_incomplete.to_csv(f"{final_folder}imports_intermediate_{base}.csv", sep=";", index=False)
        df_imports_final_incomplete.to_csv(f"{final_folder}imports_final_demand_{base}.csv", sep=";", index=False)
        df_production_incomplete.to_csv(f"{final_folder}production_{base}.csv", sep=";", index=False)


    print(f"\n>>> Global files (Years as columns) saved to: {final_folder}")


def run(base, years, metrics_wanted, paths,
        version=None,
        mapping_iso3_to_iso2=None,
        save_csv=True):
    """ Main function: loops over `years`, parses/aggregates the `base`
    (via parse_mrio_for_year or parse_gtap_for_year depending on the case),
    computes the requested metrics (`metrics_wanted`, or every available
    metric if "ALL"), saves the CSVs (metrics + exports/imports/production)
    if save_csv=True, and returns a dict {year: {metric: DataFrame}}. """

    ALL_METRICS = {
            "FDI": ["rd_dir_fdips"],
            "Patents_Citation": ["rd_dir_cit"],
            "EXIOBASE": ["rd_embod_CI", "rd_imp_CI", "rd_dir_CI"],
            "FIGARO": ["rd_embod_CI", "rd_imp_CI", "rd_dir_CI"],
            "OECD_ICIO": ["rd_embod_CI", "rd_imp_CI", "rd_dir_CI"],
            "CFC": ["rd_embod_CFC", "rd_dir_CFC"],
            "GTAP": ["rd_embod_CI", "rd_imp_CI", "rd_dir_CI"],
        }

    if metrics_wanted == "ALL":
        metrics_wanted = ALL_METRICS.get(base, [])
        print(f"ALL metrics calculated for {base} : {metrics_wanted}")

    if DIMENSION == "(c,s,t)":
        techs = tech_specs(TECHS)
    elif DIMENSION == "(c,s)":
        techs = {None: None}
    else:
        raise ValueError(f"DIMENSION must be '(c,s)' or '(c,s,t)', got {DIMENSION!r}")

    rd_stocks = {label: compute_rd_stock(paths["rd"], paths["agg_rd"], tech=patterns) for label, patterns in techs.items()}
    print(f"R&D data: {RD_UNIT} {DIMENSION} -> {paths['rd']}")
    for label in rd_stocks:
        print(f"  results written under: {output_path(paths, label)}" + ("" if label is None else f"  (technology {label}: {techs[label]})"))

    available = set.intersection(*[set(stock.index) for _, stock in rd_stocks.values()])
    skipped = [y for y in years if y not in available]
    if skipped:
        print(f"  [warning] years without R&D stock for this dimension (skipped): {skipped} | available: {min(available)}-{max(available)}")
    years = [y for y in years if y in available]

    output = {}

    if base == "FDI":
        dict_reg_gtap_to_rd  = load_mapping(paths["map_fdi"], "Region", "gtap_region", "aggregation")
        gtap_to_nace_rd_data = load_mapping(paths["map_fdi"], "Sector", "gtap_sector", "aggregation_nace")
        fdips_reg = load_fdi_static(paths["har"], dict_reg_gtap_to_rd, gtap_to_nace_rd_data)

        for year in years:
            for label, (df_rd, stock) in rd_stocks.items():
                metrics = metric_fdi_for_year(fdips_reg, stock, year)
                _store(output, year, label, metrics)
                _save(metrics, base, year, paths, save_csv, out=output_path(paths, label))
                print(f"FDI {year}{_tag(label)} : {list(metrics.keys())} calculated")

    elif base == "Patents_Citation":
        dict_sect = load_mapping(
            paths["map_citation"], "Sector", "citation_code", "aggregation_nace"
        )
        dict_reg = load_mapping(
            paths["map_citation"], "Region", "citation_region", "aggregation"
        )

        for year in years:
            df_shares = load_citation_static(
                paths["citation"], dict_sect, dict_reg, year
            )
            for label, (df_rd, stock) in rd_stocks.items():
                metrics = metric_citation_for_year(df_shares, stock, year)
                _store(output, year, label, metrics)
                _save(metrics, base, year, paths, save_csv, out=output_path(paths, label))
                print(
                    f"Patents_Citation {year}{_tag(label)} : {list(metrics.keys())} calculated with Time dimension"
                )

    elif base in ("EXIOBASE", "FIGARO", "OECD_ICIO", "CFC", "GTAP"):
        if base == "CFC":
            mapping_iso3_to_iso2 = load_mapping(paths["map_cfc_iso"], "ISO", "iso3", "iso2")

        for y in years:
            if base == "GTAP":
                if y <= 2004:
                    gtap_year = 2004
                elif 2004 < y <= 2007:
                    gtap_year = 2007
                elif 2007 < y <= 2011:
                    gtap_year = 2011
                elif 2011 < y <= 2014:
                    gtap_year = 2014
                elif 2014 < y <= 2017:
                    gtap_year = 2017
                elif 2017 < y <= 2019:
                    gtap_year = 2019
                else:
                    gtap_year = 2023

                db = parse_gtap_for_year(gtap_year, paths)
                print(f"GTAP {y} (period {gtap_year}) : parsed and aggregated IOT data")
            else:
                mrio_base = "EXIOBASE" if base == "CFC" else base
                db = parse_mrio_for_year(mrio_base, y, paths, version=version)
                print(f"{base} {y} : parsed and aggregated IOT data")

            # --- exports / imports / production by (Region, Sector) ---
            exports, imports_intermediate, imports_final_demand, production = compute_exports_imports_by_sector(db)

            if save_csv:
                _save_trade(exports, imports_intermediate, imports_final_demand, production, base, y)

            print(f"{base} {y} : exports/imports/production calculated")

            # R&D extension: one pass per technology (a single pass without technology dimension); the MRIO base itself
            # (parsing, aggregation, exports/imports/production) is done once per year
            mrio_metrics = {"rd_embod_CI", "rd_imp_CI", "rd_dir_CI", "intensity_vector", "HHI"}
            wanted_mrio  = [m for m in metrics_wanted if m in mrio_metrics]
            cfc_metrics  = {"rd_embod_CFC", "rd_dir_CFC"}
            wanted_cfc   = [m for m in metrics_wanted if m in cfc_metrics] if base == "CFC" else []
            df_cap_mapped = load_kbar_for_year(y, paths, mapping_iso3_to_iso2) if wanted_cfc else None
            shared = {}   # matrices that do not depend on the R&D stock (Leontief inverse...), reused by every technology

            for label, (df_rd, stock) in rd_stocks.items():
                rd_ext, units = build_rd_extensions_for_year(stock, df_rd, y, db.E.columns, db.units["Satellite account"])
                extended_db = db.add_extensions(io=rd_ext, matrix="E", units=units, inplace=False)

                if {"rd_embod_CI", "rd_imp_CI"} & set(wanted_mrio) and "w" not in shared:
                    shared["w"] = extended_db.w   # Leontief inverse: same for every technology
                metrics = compute_mrio_metrics(extended_db, wanted_mrio, w=shared.get("w")) if wanted_mrio else {}

                if wanted_cfc:
                    metrics.update(compute_cfc_metrics(db, extended_db, df_cap_mapped, stock, y, wanted_cfc, cache=shared))

                _store(output, y, label, metrics)
                _save(metrics, base, y, paths, save_csv, out=output_path(paths, label))
                print(f"{base} {y}{_tag(label)} : {list(metrics.keys())} saved to {output_path(paths, label)}_{y}")

            del db, extended_db
            if base != "GTAP" and base != "OECD_ICIO" and os.path.exists(paths["delete"]):
                shutil.rmtree(paths["delete"])
                print(f"Temporary file deleted : {paths['delete']}")

        # --- SAVING AND MULTI-YEAR PIVOTING ---
        if save_csv and all_exports:
            for label in rd_stocks:   # the same files in every output folder (technologies share the MRIO base)
                _write_global_files(f"{output_path(paths, label)}/", base)

    else:
        raise ValueError(f"Unmanaged database : {base}")

    print("All metrics have been calculated and saved.")
    return output


def _save(metrics, base, year, paths, save_csv, out=None):
    """ Saves every metric computed for a given year to CSV, into an
    {output}_{year}/ subfolder (out replaces paths["output"], e.g. one folder per technology). Does nothing if save_csv=False. """
    if not save_csv:
        return
    folder = f"{out or paths['output']}_{year}/"
    os.makedirs(folder, exist_ok=True)
    for name, mat in metrics.items():
        mat = mat.sort_index().sort_index(axis=1)
        mat.to_csv(f"{folder}{name}_{base}_{year}.csv", sep=";")


# %% Main script execution
if __name__ == "__main__":
    results = run(
        base=BASE,
        years=YEARS,
        metrics_wanted=METRICS_WANTED,
        paths=PATHS,
        version=VERSION_EXIO
    )

ALL metrics calculated for EXIOBASE : ['rd_embod_CI', 'rd_imp_CI', 'rd_dir_CI']
R&D data: value (c,s) -> C:/Users/oscar/Desktop/ENTICE/rendu_fin_stage/RDV (c,s).csv
  results written under: C:/Users/oscar/Desktop/ENTICE/rendu_fin_stage/SRDV (c,s)/EXIOBASE/output_loop_EXIOBASE


TypeError: download_exiobase3() got an unexpected keyword argument 'path'